# Import the modules

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Defining the Hardware for training and inference

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


#  Data Loading and Pre-Processing

In [4]:
transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307), (0.3081))
])

In [5]:
train_images = torchvision.datasets.MNIST(root='./data', train=True, transform=transforms, download=True)
test_images = torchvision.datasets.MNIST(root='./data', train=False, transform=transforms, download=True)

In [6]:
train_data = DataLoader(train_images, batch_size=64, shuffle=True)
test_data = DataLoader(test_images, batch_size=60000, shuffle=False)

# Model building

In [ ]:
class MNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(784, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )
        
    def forward(self, x):
        x = self.flatten(x)
        x = self.layers(x)
        return x

# Initialize model, loss function and Optimizer

In [ ]:
model = MNIST()

loss_function = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train(model, data, optimizer, loss_function, device):
    model = model.to(device)

    model.train()

    for image, label in data:
        image, label = image.to(device), label.to(device)
        optimizer.zero_grad()

        output = model(image)
        loss = loss_function(output, label)
        loss.backward()
        optimizer.step()
        
    return model, loss.item()


# Inference

In [ ]:
def predict(model, data):

    model.eval()
    
    with torch.no_grad():
        output = model(data)
        print(f"output shape: {output.shape}")
        _, index = output.max(1)
        print(f"model's output = {output}")
        print(f"model's prediction {index.item()}")
        
        

# Training Loop

In [ ]:
num_epoch = 5

In [ ]:
for epoch in range(num_epoch):

    model, loss = train(model, train_data, optimizer, loss_function, device)
    print(loss)

# Test the model

In [ ]:
label_idx = 63


for image, label in test_data:
    print(image.shape)
    plt.imshow(image[label_idx].squeeze(0), cmap='gray')
    print(f"actual image = {label[label_idx]}")
    predict(model, image[label_idx])
    break

# Saving the model weights

In [ ]:
torch.save(model.state_dict(), "MNIST_model.pth")

In [10]:
state = torch.load("MNIST_model.pth")

In [11]:
class MNISTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(784, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

In [12]:
evalmodel = MNISTModel()

In [13]:
evalmodel.load_state_dict(state)

<All keys matched successfully>

In [ ]:
! pip install torchmetrics

In [14]:
import torchmetrics

In [15]:
accuary = torchmetrics.Accuracy(task="multiclass", num_classes=10, average="macro").to(device)
recall = torchmetrics.Recall(task="multiclass", num_classes=10, average="macro").to(device)
precision = torchmetrics.Precision(task="multiclass", num_classes=10, average="macro").to(device)
f1 = torchmetrics.F1Score(task="multiclass", num_classes=10, average="macro").to(device)

In [16]:
evalmodel.eval()
for image, true_label in test_data:
    preds = evalmodel.forward(image)
    print(f"Accuracy {accuary(preds, true_label).item() * 100}")
    print(f"Recall {recall(preds, true_label).item() * 100}")
    print(f"Precision {precision(preds, true_label).item() * 100}")
    print(f"F1 Score {f1(preds, true_label).item() * 100}")

Accuracy 97.46145009994507
Recall 97.46145009994507
Precision 97.47502207756042
F1 Score 97.46289253234863
